# ANKIT LLM v0.1 — **one-click** Colab T4 run (autopilot)

Tumhe kuch configure nahi karna. Bas **3 cells** chalao.

1. **Cell 1** — GPU check (T4 hona chahiye: *Runtime → Change runtime type → T4 GPU*)
2. **Cell 2** — Drive mount **+ account check**. Ek Google login popup aayega (OK dabana).
   Ye cell print karega: **DRIVE ACCOUNT ka email** aur `checkpoints/t4-full` me kya hai.
   `ankit-llm/WHOAMI.txt` bhi likhta hai, taaki baad me pata chale kaun sa account chala.
   > Agar account galat hai (doosra Gmail), training chalti rahegi par checkpoints **us
   > doosre account** ki Drive me jayenge. Isliye ye line zaroor dekho.
3. **Cell 3** — autopilot, **60 minute ka chunk**. Khud dekhta hai Drive par checkpoint
   hai ya nahi (nahi → poora pipeline, hai → seedha `--mode resume`). Chunk khatam hone
   par checkpoint Drive par chala jata hai aur cell ruk jata hai — phir yahi cell dobara
   chalao, training usi step se aage badhegi.

**Checkpoints ek rolling `latest.pt` par jaate hain**, background thread se — training
Drive ka wait nahi karti, Drive ka trash nahi badhta, aur `heartbeat.json` har 100 steps
par batata hai training kahan tak pahunchi. Runtime band ho jaye to yahi cell dobara
chalao: newest checkpoint Drive se wapas aata hai.

Target: **64,000 steps = ~524M tokens = ~21.8 epochs** (corpus 24.1M tokens hai — 2B token
ka matlab 83 epochs hota, jo ratne jaisa hai).

**Kitne chunks?** 60 min ≈ ~5,000 steps ⇒ **~12–13 chunks**, har ek safe: ek chunk gaya to
sirf 60 min gaya. Data (45 MB) pehli baar Drive par cache ho jata hai, isliye aage ke chunks
me 10–20 min ka rebuild nahi hota.

In [ ]:
!nvidia-smi
import torch, platform
print("py", platform.python_version(), "| torch", torch.__version__)
assert torch.cuda.is_available(), "GPU nahi mila: Runtime -> Change runtime type -> T4 GPU"
p = torch.cuda.get_device_properties(0); cap = torch.cuda.get_device_capability(0)
print("GPU:", p.name, "| cc", f"{cap[0]}.{cap[1]}", "| VRAM", round(p.total_memory/1e9,1), "GB")
print("bf16 accelerated:", cap[0] >= 8, "(T4 -> False, isliye fp16 + GradScaler use hoga)")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, os
DRIVE = "/content/drive/MyDrive/ankit-llm"
os.makedirs(f"{DRIVE}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE}/reports", exist_ok=True)

# ---- SABSE ZAROORI CHECK: kaun sa Google account? ---------------------------
# Drive mount jo account use karta hai, wahi checkpoint ka ghar hai. Agar Colab me
# koi doosra account logged in hai to training chalti rahegi aur checkpoints *us*
# account ki Drive me jayenge - yahan kuch nahi dikhega. Isliye account print karte
# hain aur Drive me WHOAMI.txt likh dete hain (is file se pata chalta hai kaun chala).
def drive_identity():
    try:
        from googleapiclient.discovery import build
        return build("drive", "v3").about().get(fields="user").execute().get("user", {})
    except Exception as exc:
        return {"error": f"{type(exc).__name__}: {exc}"}

who = drive_identity()
print("=" * 70)
print("DRIVE ACCOUNT:", who.get("emailAddress"), "|", who.get("displayName"))
if not who.get("emailAddress"):
    print("  account pata nahi chala ->", who.get("error"))
print("My Drive top level:", sorted(os.listdir("/content/drive/MyDrive"))[:12])
ck = f"{DRIVE}/checkpoints/t4-full"
print("checkpoints/t4-full:", sorted(os.listdir(ck))[:12] if os.path.isdir(ck) else "MISSING")
print("=" * 70)
with open(f"{DRIVE}/WHOAMI.txt", "w", encoding="utf-8") as fh:
    fh.write(json.dumps(who, indent=2) + "\n")
print("drive ready:", DRIVE)

In [ ]:
# ⬇️ यही एक cell चलाना है (Run ▶). बाकी सब अपने आप होगा.
import os, subprocess, sys

REPO     = "https://github.com/pushparaj9749/tool.git"
REPO_DIR = "/content/ankit-llm"
DRIVE    = "/content/drive/MyDrive/ankit-llm"

def run(cmd, **kw):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, **kw)

if not os.path.isdir(REPO_DIR):
    run(["git", "clone", REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)

# --- wo build chuno jisme teeno fixes hain -----------------------------------
# 1. checkpoint device-fix       -> training/checkpoint.py: _to_cpu_copy
# 2. OOM self-heal / VRAM budget -> scripts/colab_autopilot.py: VRAM_BUDGET_FRACTION
# 3. async Drive mirror          -> training/mirror.py: BackgroundMirror
run(["git", "fetch", "origin"], check=False, capture_output=True)
CANDIDATES = ["main", "arena/87318948-tool"]
# 4. rolling mirror (quota leak fix) + heartbeat/watchdog
MARKERS = {"training/checkpoint.py": "_to_cpu_copy",
           "scripts/colab_autopilot.py": "_preflight_mirror",
           "training/mirror.py": "LATEST_CKPT_NAME",
           "training/pretrain.py": "_watch_mirror"}

def build_ok() -> tuple[bool, str]:
    if not os.path.exists("scripts/colab_autopilot.py"):
        return False, "autopilot missing"
    missing = []
    for path, marker in MARKERS.items():
        txt = open(path, encoding="utf-8").read() if os.path.exists(path) else ""
        if marker not in txt:
            missing.append(f"{path}:{marker}")
    return (not missing), ("ok" if not missing else "missing " + ", ".join(missing))

chosen = None
for ref in CANDIDATES:
    r = run(["git", "checkout", "-B", "run", f"origin/{ref}"],
            check=False, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"[skip] origin/{ref}: {(r.stderr or '').strip().splitlines()[:1]}")
        continue
    ok, why = build_ok()
    print(f"[{'use' if ok else 'skip'}] origin/{ref}: {why}")
    if ok:
        chosen = ref
        break
assert chosen, ("koi bhi ref async-mirror fix ke saath nahi mila — GitHub par PR "
                "merge karna padega")
print("code build:", chosen, "|", run(["git", "rev-parse", "--short", "HEAD"],
      capture_output=True, text=True).stdout.strip())
# data/ aur checkpoints/ gitignored hain -> checkout inhe chhoota nahi

# Colab ka apna torch mat chhedo — sirf chhoti dependencies (pynvml = GPU utilisation)
run([sys.executable, "-m", "pip", "-q", "install",
     "pyyaml", "tokenizers", "numpy", "pynvml", "psutil", "pytest"], check=True)

# --- pehli baar ya resume? Drive dekh kar khud decide hota hai ----------------
def drive_checkpoint_exists() -> bool:
    d = os.path.join(DRIVE, "checkpoints", "t4-full")
    return os.path.isdir(d) and any(f.endswith(".pt") for f in os.listdir(d))

RESUMING = drive_checkpoint_exists()
MODE = "resume" if RESUMING else "all"
print(f"\n>>> MODE = {MODE}  "
      f"({'Drive par checkpoint mila -> seedha training resume' if RESUMING else 'checkpoint nahi -> poora pipeline (bench+pilot+train)'})")

# pichhle chunk ka progress (agar hai)
_prog = os.path.join(DRIVE, "reports", "autopilot_chunk_progress.json")
if os.path.exists(_prog):
    try:
        _p = json.load(open(_prog))
        print(f">>> PROGRESS: step {_p.get('step'):,} / {_p.get('max_steps'):,} "
              f"({_p.get('percent')}%) — {_p.get('updated_at')}")
    except Exception as _exc:
        print("progress file padha nahi ja saka:", _exc)

# --- CHHOTA CHUNK: 60 minute max -------------------------------------------
# Har chunk ke baad final checkpoint Drive par chala jata hai aur cell ruk jata hai.
# Chunk khatam = bas yahi cell dobara chalao; training usi step se aage badhti hai.
# 60 min = ~53 min training + ~7 min startup/save/flush = ~5,000 steps per chunk.
SESSION_BUDGET_MIN = 60
BUILD = "t4fix"   # build-tag: isse pata chalta hai kaunsa code chal raha hai

# Output LIVE Drive me likha jayega — session crash ho jaye tab bhi log bacha rahega
import time
LOG_DIR = os.path.join(DRIVE, "reports")
os.makedirs(LOG_DIR, exist_ok=True)
LOG = os.path.join(LOG_DIR, f"autopilot_{BUILD}_{time.strftime('%m%d_%H%M')}.log")
cmd = [sys.executable, "-u", "scripts/colab_autopilot.py",
       "--mode", MODE, "--session-budget-min", str(SESSION_BUDGET_MIN)]
env = dict(os.environ, PYTHONUNBUFFERED="1")
print("log ->", LOG, "\n", flush=True)
with open(LOG, "w", encoding="utf-8") as fh:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
        fh.write(line); fh.flush()
rc = proc.wait()
print(f"\nreturncode = {rc}  |  log saved: {LOG}")
# --- kis account ki Drive me likh rahe hain? (log me bhi record rahe) ---------
try:
    import json as _json
    _who = _json.load(open(os.path.join(DRIVE, "WHOAMI.txt")))
    print("DRIVE ACCOUNT:", _who.get("emailAddress"), "|", _who.get("displayName"))
except Exception as _exc:
    print("WHOAMI padha nahi ja saka:", _exc)

ck = os.path.join(DRIVE, "checkpoints", "t4-full")
print("\n>>> CHUNK KHATAM. Drive par ab tak ke checkpoints:")
if os.path.isdir(ck):
    for f in sorted(os.listdir(ck)):
        print("   ", f)
else:
    print("    (folder hi nahi bana — Drive mount / account check karo)")
print("\n>>> Agla chunk: yahi Cell 3 dobara chalao (Run). Wo khud resume karega.")

## Result dekho

`reports/autopilot_summary.md` (Drive me bhi copy hota hai) me sab measured numbers honge:
benchmark table, auto-selected config, pilot verification, checkpoints, mirror stats.

**Yahi file ka content mujhe bhej dena** — usme sab kuch hoga.

## Chat try karo (GPU par fast hai)

```python
from inference.conversation import ChatEngine
eng = ChatEngine.from_checkpoint("checkpoints/t4-sft/final.pt", device="cuda")
print(eng.chat_once([{"role":"user","content":"नमस्ते अंकित"}], mode="adaptive"))
```

## Agli session (training continue)

Wahi cell 3 dobara chalao. Cell khud `--mode resume` chunta hai: newest checkpoint
Drive se local disk par restore hota hai, phir `--resume auto` usi LR schedule par aage
badhta hai. Sirf pichhle checkpoint ke baad ke kuch minute jaate hain.